# NOTEBOOK 04 — EXPLORATORY DATA ANALYSIS & VISUALIZATION

### Mục tiêu
EDA phải tạo ra insight có số liệu hỗ trợ và làm nền cho Feature Engineering.

Các câu hỏi chính:

1. Nhiệt độ 5 thành phố phân bố khác nhau thế nào?
2. Seasonality có giống nhau giữa Bắc và Nam bán cầu không?
3. Xu hướng dài hạn là bao nhiêu °C/thập kỷ?
4. Temperature anomaly so với baseline thay đổi thế nào?
5. Độ bất định của dữ liệu thay đổi ra sao theo lịch sử?
6. Context Country/Global/Country Summary/State Summary liên hệ với nhiệt độ thành phố thế nào?

## I. Setup và đọc dữ liệu sạch

In [ ]:
from pathlib import Path
import os

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
ARTIFACTS = PROJECT_ROOT / "artifacts"
APP_DIR = PROJECT_ROOT / "app"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
ARTIFACTS.mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

url_object = URL.create(
    "postgresql",
    username="postgres",
    password="123456",
    host="localhost",
    port=5432,
    database="climate_change_db1",
)
engine = create_engine(url_object, pool_pre_ping=True)

with engine.connect() as conn:
    print("Database:", conn.execute(text("SELECT current_database()")).scalar())

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from IPython.display import display

df = pd.read_sql(
    text("SELECT * FROM climate_country_clean ORDER BY country, dt"),
    engine,
    parse_dates=["dt"]
)

df["year"] = df["dt"].dt.year
df["month"] = df["dt"].dt.month

print("Shape:", df.shape)
print("Countries:", sorted(df["country"].dropna().unique()))

## II. Target Analysis

Phân bố target dùng dữ liệu **quan sát thật** (`target_is_observed = 1`), không trộn nhãn nội suy.

In [ ]:
observed = df[df["target_is_observed"] == 1].copy()

fig, ax = plt.subplots(figsize=(11, 5))
sns.histplot(
    data=observed,
    x="average_temperature_observed",
    hue="country",
    bins=50,
    element="step",
    stat="density",
    common_norm=False,
    ax=ax
)
ax.set_title("Observed temperature distribution by city")
ax.set_xlabel("Average temperature (°C)")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=observed,
    x="country",
    y="average_temperature_observed",
    ax=ax
)
ax.set_title("Observed temperature by city")
ax.tick_params(axis="x", rotation=20)
plt.show()

## III. Seasonality theo tháng

Không được kết luận “tháng 8 nóng nhất” cho tất cả 5 thành phố. Sydney ở Nam bán cầu nên seasonal phase khác.

In [ ]:
monthly = (
    df.groupby(["country", "month"])["average_temperature_filled"]
      .mean()
      .reset_index()
)

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(
    data=monthly,
    x="month",
    y="average_temperature_filled",
    hue="country",
    marker="o",
    ax=ax
)
ax.set_title("Monthly seasonality")
ax.set_ylabel("Average temperature (°C)")
plt.show()

hottest = (
    monthly.loc[
        monthly.groupby("country")["average_temperature_filled"].idxmax(),
        ["country", "month", "average_temperature_filled"]
    ]
    .rename(columns={
        "month": "hottest_month",
        "average_temperature_filled": "hottest_month_temp"
    })
)

coldest = (
    monthly.loc[
        monthly.groupby("country")["average_temperature_filled"].idxmin(),
        ["country", "month", "average_temperature_filled"]
    ]
    .rename(columns={
        "month": "coldest_month",
        "average_temperature_filled": "coldest_month_temp"
    })
)

season_summary = hottest.merge(coldest, on="country")
season_summary["seasonal_amplitude_C"] = (
    season_summary["hottest_month_temp"]
    - season_summary["coldest_month_temp"]
)

display(season_summary)

## IV. Annual Mean và Rolling 10 năm

Monthly temperature có seasonal noise lớn.  
Để nhìn long-term trend, aggregate theo năm và dùng rolling mean 10 năm.

In [ ]:
annual = (
    df.groupby(["country", "year"])["average_temperature_filled"]
      .mean()
      .reset_index()
)

annual["rolling_10y"] = (
    annual.groupby("country")["average_temperature_filled"]
          .transform(lambda s: s.rolling(10, min_periods=5).mean())
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(
    data=annual,
    x="year",
    y="rolling_10y",
    hue="country",
    ax=ax
)
ax.set_title("10-year rolling mean temperature")
ax.set_ylabel("°C")
plt.show()

## V. Định lượng trend °C/thập kỷ

Tính regression slope trên annual mean từ 1900 trở đi.

In [ ]:
trend_rows = []

for country, group in annual[annual["year"] >= 1900].groupby("country"):
    g = group.dropna(subset=["average_temperature_filled"])
    fit = linregress(g["year"], g["average_temperature_filled"])

    trend_rows.append({
        "country": country,
        "trend_C_per_decade": fit.slope * 10,
        "p_value": fit.pvalue,
        "r_squared": fit.rvalue ** 2
    })

trend_df = pd.DataFrame(trend_rows).sort_values(
    "trend_C_per_decade",
    ascending=False
)
display(trend_df)

## VI. Temperature Anomaly

Baseline dùng **1901–1930**, tính riêng cho `city + month`.  
Điều này loại phần lớn seasonality trước khi so sánh mức nóng/lạnh bất thường.

In [ ]:
BASELINE_START = 1901
BASELINE_END = 1930

baseline = (
    df[df["year"].between(BASELINE_START, BASELINE_END)]
    .groupby(["country", "month"])["average_temperature_filled"]
    .mean()
    .rename("baseline_temperature")
    .reset_index()
)

anom = df.merge(
    baseline,
    on=["country", "month"],
    how="left"
)

anom["temperature_anomaly"] = (
    anom["average_temperature_filled"]
    - anom["baseline_temperature"]
)

annual_anom = (
    anom.groupby(["country", "year"])["temperature_anomaly"]
        .mean()
        .reset_index()
)

annual_anom["rolling_10y_anomaly"] = (
    annual_anom.groupby("country")["temperature_anomaly"]
               .transform(
                   lambda s: s.rolling(10, min_periods=5).mean()
               )
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(
    data=annual_anom,
    x="year",
    y="rolling_10y_anomaly",
    hue="country",
    ax=ax
)
ax.axhline(0, linestyle="--", linewidth=1)
ax.set_title(
    f"Temperature anomaly vs {BASELINE_START}-{BASELINE_END} baseline"
)
ax.set_ylabel("Anomaly (°C)")
plt.show()

## VII. Uncertainty theo thời gian

In [ ]:
uncertainty_year = (
    df.groupby(["country", "year"])["average_temperature_uncertainty"]
      .mean()
      .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.lineplot(
    data=uncertainty_year,
    x="year",
    y="average_temperature_uncertainty",
    hue="country",
    ax=ax
)
ax.set_title("Measurement uncertainty over time")
ax.set_ylabel("Uncertainty (°C)")
plt.show()

## VIII. Context từ 4 bảng phụ

Phân tích country target với:
- Country
- Global
- Country-summary theo country
- State-summary theo country

In [ ]:
context_cols = [
    "average_temperature_filled",
    "country_temperature",
    "global_temperature",
    "city_country_avg_temperature",
    "state_country_avg_temperature",
    "average_temperature_uncertainty",
    "country_uncertainty",
    "global_uncertainty"
]

corr = df[context_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    ax=ax
)
ax.set_title("Correlation: country target and auxiliary context")
plt.show()

In [ ]:
df["city_minus_country"] = (
    df["average_temperature_filled"]
    - df["country_temperature"]
)
df["country_minus_global"] = (
    df["country_temperature"]
    - df["global_temperature"]
)

gap_summary = (
    df.groupby("country")[
        ["city_minus_country", "country_minus_global"]
    ]
    .agg(["mean", "std"])
)
display(gap_summary)

## IX. 3+ Insight định lượng

Cell dưới tạo bảng số liệu để nhóm viết insight dựa trên kết quả thật.

In [ ]:
latest_anomaly = (
    annual_anom.sort_values(["country", "year"])
               .groupby("country")
               .tail(10)
               .groupby("country")["temperature_anomaly"]
               .mean()
               .rename("mean_anomaly_last10y_C")
               .reset_index()
)

insight_table = (
    trend_df.merge(
        season_summary[
            ["country", "hottest_month", "coldest_month", "seasonal_amplitude_C"]
        ],
        on="country"
    )
    .merge(latest_anomaly, on="country", how="left")
)

display(insight_table)

print("\nDATA-DRIVEN INSIGHTS")
for _, row in insight_table.iterrows():
    print(
        f"- {row['country']}: trend {row['trend_C_per_decade']:.3f} °C/decade; "
        f"hottest month={int(row['hottest_month'])}; "
        f"coldest month={int(row['coldest_month'])}; "
        f"seasonal amplitude={row['seasonal_amplitude_C']:.2f} °C; "
        f"mean anomaly last 10y={row['mean_anomaly_last10y_C']:.2f} °C."
    )

## X. EDA → Feature Engineering

| Insight | Feature đề xuất |
|---|---|
| Seasonality 12 tháng | `month_sin`, `month_cos` |
| Sydney ngược pha Bắc bán cầu | latitude, hemisphere |
| Trend dài hạn khác nhau theo country | `time_idx × city` |
| Country/Global context có quan hệ | climatology từ bảng phụ |
| Uncertainty thay đổi | climatology uncertainty |
| Gap city-country/global | context-gap features |

Không dùng `average_temperature_filled` của tháng hiện tại làm input model.

In [ ]:
eda_summary = {
    "baseline_period": [BASELINE_START, BASELINE_END],
    "trend": json.loads(trend_df.to_json(orient="records")),
    "seasonality": json.loads(season_summary.to_json(orient="records")),
    "latest_anomaly": json.loads(latest_anomaly.to_json(orient="records"))
}

(ARTIFACTS / "eda_summary.json").write_text(
    json.dumps(eda_summary, indent=2, ensure_ascii=False),
    encoding="utf-8"
)
print("Saved artifacts/eda_summary.json")

## XI. Kết luận Notebook 04

Phần kết luận trong báo cáo phải có ít nhất 3 insight với con số cụ thể, không chỉ mô tả hình.

Ví dụ loại nhận xét được chấp nhận:
- country nào có trend °C/decade cao hơn;
- Sydney có hottest month khác nhóm Bắc bán cầu;
- anomaly 10 năm cuối khác baseline bao nhiêu;
- uncertainty lịch sử thay đổi thế nào.

Không dùng Prophet/model forecast để “chứng minh” nguyên nhân biến đổi khí hậu.